# Chapter 12 — C and C++ (translated to Python concepts)

This chapter is language-specific. For each question we:
1. State the C/C++ answer briefly.
2. Show the **Python analogue**, meaning the equivalent concept or mechanism. Where it is instructive, we also go below Python using `ctypes` to touch raw memory.

C++ concepts and their Python counterparts:

| C++ | Python |
|---|---|
| classes, inheritance | classes, MRO |
| `virtual` functions | every method is "virtual" (dynamic dispatch) |
| constructors / destructors | `__init__` / `__del__` (and context managers) |
| copy constructor | `copy.copy` / `copy.deepcopy`, `__copy__` / `__deepcopy__` |
| templates | duck typing + `typing.Generic` |
| pointers / references | every name is a reference; `id()` ~ address |
| smart pointers | CPython's reference counting (`sys.getrefcount`, `weakref`) |

In [1]:
import copy
import ctypes
import gc
import sys
import weakref
from collections import deque

## 12.1 Last K Lines
Print the last K lines of a file. In C++, use a circular array of K strings. In Python, `deque(maxlen=K)` *is* that circular buffer.

In [2]:
import os
import tempfile


def last_k_lines(path: str, k: int) -> list[str]:
    with open(path) as f:
        return list(deque(f, maxlen=k))


def last_k_lines_circular(path: str, k: int) -> list[str]:
    """Explicit circular array, as in the C++ solution."""
    buf: list[str | None] = [None] * k
    n = 0
    with open(path) as f:
        for line in f:
            buf[n % k] = line
            n += 1
    start = n % k if n >= k else 0
    count = min(n, k)
    return [buf[(start + i) % k] for i in range(count)]  # type: ignore[misc]


fd, path = tempfile.mkstemp(text=True)
with os.fdopen(fd, "w") as f:
    f.writelines(f"line {i}\n" for i in range(1, 101))
assert last_k_lines(path, 3) == last_k_lines_circular(path, 3) == ["line 98\n", "line 99\n", "line 100\n"]
assert last_k_lines_circular(path, 500)[0] == "line 1\n"
print("12.1 ok")

12.1 ok


## 12.2 Reverse String
In C, reverse a null-terminated string in place with two pointers. Python strings are immutable, so to get the true in-place version we use a mutable `ctypes` char buffer that really is null-terminated.

In [3]:
def reverse_c_string(buf: ctypes.Array) -> None:
    end = 0
    while buf[end] != b"\x00":   # find the null terminator
        end += 1
    start, end = 0, end - 1
    while start < end:
        buf[start], buf[end] = buf[end], buf[start]
        start += 1
        end -= 1


cbuf = ctypes.create_string_buffer(b"hello world")
reverse_c_string(cbuf)
assert cbuf.value == b"dlrow olleh"
assert "hello"[::-1] == "olleh"  # idiomatic Python
print("12.2 ok")

12.2 ok


## 12.3 Hash Table vs. STL Map
A hash table has `O(1)` average lookup and **no ordering**. An STL `map` is a balanced BST (red-black tree) with `O(log n)` lookup, and it **iterates in key order**. Python's `dict` is a hash table (insertion-ordered since 3.7). For a sorted map, use `bisect` over sorted keys (or `sortedcontainers.SortedDict`). A minimal sorted map:

In [4]:
import bisect


class SortedMap:
    """std::map-like: ordered iteration, floor/ceiling queries. O(n) insert here; a tree gives O(log n)."""

    def __init__(self) -> None:
        self._keys: list = []
        self._vals: dict = {}

    def __setitem__(self, k, v) -> None:
        if k not in self._vals:
            bisect.insort(self._keys, k)
        self._vals[k] = v

    def __getitem__(self, k):
        return self._vals[k]

    def items(self):
        return ((k, self._vals[k]) for k in self._keys)

    def floor_key(self, k):
        i = bisect.bisect_right(self._keys, k)
        return self._keys[i - 1] if i else None


sm = SortedMap()
for k in [50, 10, 30, 20]:
    sm[k] = str(k)
assert [k for k, _ in sm.items()] == [10, 20, 30, 50] and sm.floor_key(45) == 30
print("12.3 ok")

12.3 ok


## 12.4 Virtual Functions
In C++, a `virtual` function call dispatches through the object's **vtable**, so the runtime type decides which function runs. Non-virtual calls are resolved from the static type. In Python every method call is dynamic: the lookup goes through the instance's class `__mro__` at call time. We imitate the C++ distinction by calling a base implementation explicitly.

In [5]:
class Shape:
    def name(self) -> str:
        return "Shape"

    def describe(self) -> str:
        return f"I am a {self.name()}"             # 'virtual' dispatch via self

    def describe_static(self) -> str:
        return f"I am a {Shape.name(self)}"        # like a non-virtual call bound at compile time


class Circle(Shape):
    def name(self) -> str:
        return "Circle"


c = Circle()
assert c.describe() == "I am a Circle"
assert c.describe_static() == "I am a Shape"
print([k.__name__ for k in Circle.__mro__])  # the Python "vtable" lookup order

['Circle', 'Shape', 'object']


## 12.5 Shallow vs. Deep Copy
A shallow copy copies member values, including pointers, so the copy shares the referenced objects. A deep copy duplicates the objects that are pointed to. Python has `copy.copy` and `copy.deepcopy`, and classes can customize both.

In [6]:
class Team:
    def __init__(self, name: str, members: list[str]) -> None:
        self.name, self.members = name, members


t1 = Team("a", ["ann", "bob"])
shallow, deep = copy.copy(t1), copy.deepcopy(t1)
t1.members.append("cat")
assert shallow.members == ["ann", "bob", "cat"]   # shared list
assert deep.members == ["ann", "bob"]             # independent
print("12.5 ok")

12.5 ok


## 12.6 Volatile
In C, `volatile` tells the compiler a variable may change outside the current code path (hardware, signal handlers, other threads), so it must not cache the value in a register. Python has no equivalent. Every attribute read goes through the interpreter, so there's no register caching to disable. For **thread safety**, `volatile` was never the right tool in any language: use proper synchronization (`threading.Event`, locks, atomics). Here is the "flag polled by another thread" pattern done correctly:

In [7]:
import threading
import time

stop = threading.Event()
ticks = 0


def worker() -> None:
    global ticks
    while not stop.is_set():   # the classic 'volatile bool running' loop
        ticks += 1
        time.sleep(0.001)


th = threading.Thread(target=worker)
th.start()
time.sleep(0.02)
stop.set()
th.join()
assert ticks > 0
print("12.6 ok, ticks:", ticks)

12.6 ok, ticks: 18


## 12.7 Virtual Base Class
Why should a base class destructor be `virtual`? If you `delete` a derived object through a base pointer and the destructor isn't virtual, only the base destructor runs, and the derived class's resources leak. In Python, `__del__` is looked up dynamically, so the derived finalizer always runs. Derived classes must still call `super().__del__()` / `super().close()` themselves. Prefer explicit cleanup (context managers) over `__del__`.

In [8]:
log: list[str] = []


class Base:
    def __del__(self) -> None:
        log.append("Base.__del__")


class Derived(Base):
    def __del__(self) -> None:
        log.append("Derived.__del__")
        super().__del__()


obj: Base = Derived()   # "base pointer" to a derived object
del obj
gc.collect()
assert log == ["Derived.__del__", "Base.__del__"]
print(log)

['Derived.__del__', 'Base.__del__']


## 12.8 Copy Node
Write a method that takes a pointer to a node structure and returns a complete copy of the structure. The node has two pointers to other nodes, and the structure may contain cycles. Keep a map from original to copy (C++ uses `std::map<Node*, Node*>`) so shared nodes and cycles are copied exactly once. This is what `copy.deepcopy`'s `memo` dict does.

In [9]:
class Node:
    def __init__(self, val: int) -> None:
        self.val = val
        self.ptr1: "Node | None" = None
        self.ptr2: "Node | None" = None


def copy_node(n: Node | None, memo: dict[int, Node] | None = None) -> Node | None:
    if n is None:
        return None
    memo = {} if memo is None else memo
    if id(n) in memo:
        return memo[id(n)]
    clone = Node(n.val)
    memo[id(n)] = clone  # register before recursing to handle cycles
    clone.ptr1 = copy_node(n.ptr1, memo)
    clone.ptr2 = copy_node(n.ptr2, memo)
    return clone


a, b = Node(1), Node(2)
a.ptr1, a.ptr2, b.ptr1, b.ptr2 = b, a, a, b   # cycles and self-loops
ca = copy_node(a)
assert ca is not a and ca.ptr2 is ca and ca.ptr1.ptr1 is ca and ca.ptr1 is not b
assert copy.deepcopy(a).ptr2.ptr2.val == 1
print("12.8 ok")

12.8 ok


## 12.9 Smart Pointer
A smart pointer class is a template (generic) that wraps a pointer and keeps a shared **reference count**. Copying it increments the count, and destroying it decrements the count. When the count hits 0, it frees the object. CPython already does this for every object. Below is an explicit version, plus a look at the real refcounts.

In [10]:
from typing import Generic, TypeVar

T = TypeVar("T")


class SmartPointer(Generic[T]):
    def __init__(self, obj: T | None = None, _shared: list | None = None) -> None:
        # _shared = [obj, refcount] shared by all copies
        self._shared = _shared if _shared is not None else [obj, 1]

    def copy(self) -> "SmartPointer[T]":
        self._shared[1] += 1
        return SmartPointer(_shared=self._shared)

    def release(self) -> None:
        if self._shared is None:
            return
        self._shared[1] -= 1
        if self._shared[1] == 0:
            print(f"freeing {self._shared[0]!r}")
            self._shared[0] = None
        self._shared = None  # type: ignore[assignment]

    def get(self) -> T:
        return self._shared[0]

    @property
    def count(self) -> int:
        return self._shared[1]


p1 = SmartPointer({"data": 42})
p2 = p1.copy()
assert p1.count == p2.count == 2
p1.release()
assert p2.count == 1 and p2.get() == {"data": 42}
p2.release()

class Thing: ...


x = Thing()
print("CPython refcount of x:", sys.getrefcount(x) - 1)  # -1 for the getrefcount argument
y = x
print("after y = x:", sys.getrefcount(x) - 1)
w = weakref.ref(x)          # like std::weak_ptr - doesn't bump the count
del x, y
gc.collect()
assert w() is None

freeing {'data': 42}
CPython refcount of x: 1
after y = x: 2


## 12.10 Malloc
Write an aligned `malloc` and `free`: the returned address must be divisible by a power of two. Allocate `size + alignment - 1 + sizeof(void*)` bytes, round the address up, and stash the original pointer just before the aligned block so `free` can find it. We do this for real with `ctypes` and libc.

In [11]:
import ctypes.util

libc = ctypes.CDLL(ctypes.util.find_library("c"))
libc.malloc.restype = ctypes.c_void_p
libc.malloc.argtypes = [ctypes.c_size_t]
libc.free.argtypes = [ctypes.c_void_p]
PTR = ctypes.sizeof(ctypes.c_void_p)


def aligned_malloc(size: int, alignment: int) -> int:
    offset = alignment - 1 + PTR
    raw = libc.malloc(size + offset)
    if not raw:
        raise MemoryError
    aligned = (raw + offset) & ~(alignment - 1)
    ctypes.c_void_p.from_address(aligned - PTR).value = raw   # hide original pointer
    return aligned


def aligned_free(aligned: int) -> None:
    raw = ctypes.c_void_p.from_address(aligned - PTR).value
    libc.free(raw)


for align in (16, 64, 128, 4096):
    p = aligned_malloc(1000, align)
    assert p % align == 0
    ctypes.memset(p, 0xAB, 1000)          # memory is usable
    aligned_free(p)
print("12.10 ok")

12.10 ok


## 12.11 2D Alloc
Write `my2DAlloc` so that `arr[i][j]` works on the result, and minimize the number of `malloc` calls. Make **one** allocation: an array of row pointers followed by the data. Each row pointer points into the data block, and a single `free` releases everything.

In [12]:
def my_2d_alloc(rows: int, cols: int):
    header = rows * PTR
    data = rows * cols * ctypes.sizeof(ctypes.c_int)
    base = libc.malloc(header + data)
    row_ptrs = (ctypes.c_void_p * rows).from_address(base)
    for i in range(rows):
        row_ptrs[i] = base + header + i * cols * ctypes.sizeof(ctypes.c_int)
    # View as int** so arr[i][j] works
    IntRow = ctypes.POINTER(ctypes.c_int)
    return base, ctypes.cast(base, ctypes.POINTER(IntRow))


base, arr = my_2d_alloc(3, 4)
for i in range(3):
    for j in range(4):
        arr[i][j] = i * 10 + j
assert [[arr[i][j] for j in range(4)] for i in range(3)] == [[0, 1, 2, 3], [10, 11, 12, 13], [20, 21, 22, 23]]
libc.free(base)  # single free
print("12.11 ok")

12.11 ok
